## Green Hackenbush

Une position du jeu est la donnée d'un graphe fini non orienté (les arêtes multiples et les boucles sont autorisées) et d'un sous-ensemble non vide de l'ensemble des sommets appelé le *sol*. On suppose que chaque sommet est relié par un chemin à au moins un sommet situé sur le sol.

Chaque joueur, à son tour, choisit une arête et la supprime du graphe ; il supprime aussi chaque élément, sommet ou arête, qui n'est plus relié au sol. Le joueur qui ne peut plus jouer (il n'y a plus d'arête) a perdu.

Dans la figure suivante, les sommets du sol sont indiqués par des petits cercles au lieu de petits disques. Ils sont sur la droite en pointillés.

![theHackenbushEstate](images/theHackenbushEstate.png)





### Calcul du nombre de Grundy d'une figure

- Tout d'abord, on identifie en un sommet unique $r$ tous les sommets du sol.

- Tant que le graphe contient des cycles de longueur $\geqslant2$, on identifie en un sommet unique $s0$ tous les sommets d'un tel cycle et on remplace chaque arête du cycle par une boucle attachée à $s0$.

- On remplace chaque boucle attachée à un sommet $s$ par un brin attaché à $s$ (un *brin* est une arête dont une des extrémités est de degré $1$).

Le résultat de ces transformations est un arbre $T$ enraciné en le sommet $r$.

Exemple d'une figure et de sa transformée :

![girl](images/girl.png)

On associe à chaque arête $a$ de $T$ un entier $\tau(a)>0$,  appelé sa *tension*, défini par induction par la règle suivante :

$\tau(a)=\tau(a_1)\oplus\ldots\oplus\tau(a_k)+1$ où $a_1,\ldots,a_k$ sont les arêtes filles de $a$ dans l'arbre.

En particulier, comme  un brin n'a pas de fille, sa tension est $1$.

Le nombre $\tau(a_1)\oplus\ldots\oplus\tau(a_k)$ est appelé le *poids* du sommet $s$ qui relie $a$ à ses filles.
Le poids de la racine de $T$ est appelé le *poids* de la figure.

Si on marque chaque arête de $T$ par sa tension (en omettant de marquer les $1$ des brins pour ne pas surchager le dessin), on obtient pour l'exemple précédent :

![treeGirl](images/treeGirl.png)

On peut aussi marquer chaque arête de la figure initiale qui correspond à une arête de $T$ (autrement dit qui ne fait pas partie d'un cycle), ce qui donne pour notre exemple :

![girlWithStresses](images/girlWithStresses.png)

Pour chaque cycle maximal $C$ de la figure, si le nombre $n$ d'arêtes de $C$ est pair, les sommets de $C$ sont identifiés entre eux et, comme $1\oplus1=0$, on peut, pour le calcul des tensions, omettre les $n$ brins rajoutés. Si $n$ est impair, on remplace les $n$ brins par un seul brin ; on notifie ce cas  en coloriant en orange l'un des sommets de $C$. 

**Théorème de Hackenbush** La valeur (nombre de Grundy) de la figure est égale à son poids.

**preuve**  voir plus loin la démonstration de Conway $\blacksquare$
<!-- voir [ONAG](https://www.chasse-aux-livres.fr/prix/1568811276/on-numbers-and-games-john-h-conway?reloaded=1) 
p. 165 et suivantes --> 

<!-- Ceci est un commentaire -->
Admettons le théorème. 

La valeur de notre figure exemple est $8\oplus1=9\neq0$, il s'agit donc d'une position gagnante.
Pour trouver un coup gagnant, il faut s'arranger pour que l'arête verticale $a_0$ ait pour tension $1$.  
Pour cela, on fait en sorte que la nim-somme des tensions des $3$ arêtes filles de $a_0$ soit nulle. 
Comme $2\oplus1\oplus3=0$  on donne la valeur $1$ au manche du parapluie en supprimant un brin du parapluie.

![girlAfterOneStep](images/girlAfterOneStep.png)


Autre exemple : la valeur de

![theHackenbushEstateWithStresses](images/theHackenbushEstateWithStresses.png)

est $15\oplus6\oplus4\oplus3\oplus4=10$. Il y a un unique coup gagnant consistant à supprimer l'arête représentée en rouge.



### implémentation

In [1]:
import math
import json
import drawSvg as draw
from drawSvg.widgets import DrawingWidget
from collections import defaultdict
from copy import deepcopy

xmax, ymax, eps, leftMargin, epsilon = 1200, 520, 30, 100, 20 # ne pas modifier ymax - eps
nGrid, topMargin = (ymax - eps) // epsilon, (ymax - eps) % epsilon

def theta(xc, yc, r, x, y):
    d = r + x - xc
    return 2 * math.atan((y - yc) / d) if d else math.pi

class HACKENBUSH():

    def __init__(self, name = None):

        if name is None:
            self.VERTICES = []
            self.EDGES = []
            self.ARCS = []
        else:
            self.load(name)

    def load(self, name):
     
        file = f'exemplesHackenbush/{name}.json'
        with open(file) as f:
            d = json.load(f)  
            self.VERTICES = d['vertices']
            self.EDGES = list(set(map(tuple, d['edges']))) 
            self.ARCS = d['arcs']

        self.init()

    def save(self, name):
        
        d = {'vertices': self.VERTICES,
            'edges': self.EDGES,
            'arcs' : self.ARCS}
        
        file = f'exemplesHackenbush/{name}.json'
        with open(file, 'w') as f:
            json.dump(d, f)

    def init(self):

        self.vertex = list(map(tuple, self.VERTICES))
        self.index_of_vertex = {self.vertex[i] : i for i in range(len(self.vertex))}
        self.vertices = set(range(len(self.vertex))) # ensemble des indices des sommets non encore effacés
        self.ground = [i for i in range(len(self.vertex)) if self.vertex[i][1] == ymax - eps]
        self.root = self.ground[0]
        
        self.edge = list(map(lambda l: (tuple(l),None), self.EDGES)) # aretes rectilignes

        self.arc = []
        for i, (xc, yc, r, l) in enumerate(self.ARCS):
                
            L = sorted([((x, y), theta(xc, yc, r, x, y)) for x, y in l], key = lambda t: t[1])
            self.arc.append((xc, yc, r, L))
            for j, ((x, y), t) in enumerate(L):
                j1 = (j + 1) % len(L)
                x1, y1 = L[j1][0]
                self.edge.append(((x, y, x1, y1), (i, j, j1))) # aretes arcs de cercle (tout le cercle si (x, y) = (x1, y1))
            
        self.index_of_edge = {self.edge[i][0] : i for i in range(len(self.edge))}
        self.edges = set(range(len(self.edge))) #  ensemble des indices des aretes non encore effacées

        self.makeTree()
        self.G = self.g(self.root) - 1
        
    def modify(self):

        d = draw.Drawing(xmax + leftMargin, ymax)
        d.append(draw.Line(0, ymax - eps, xmax, ymax - eps, stroke_width = 1, stroke='green', stroke_dasharray='5,2'))
        
        group = draw.Group()
        d.append(group)

        last_edge = False
        last_circle = False
        a = None
        b = None

        lastChanges = []
        LINE, CIRCLE = 0, 1
        mode = LINE
        grid = False

        def drawButton(rectangle, text, extText = '', color = 'blue'):
            x, y, w, h = rectangle
            group.draw(draw.Text(text, 12, x + w // 2, y + h // 2, stroke = color, center = True, stroke_width=.3))
            group.draw(draw.Text(extText, 12, x + 2 * w, y + h // 2, stroke = color, center = True, stroke_width=.3))
            group.draw(draw.Rectangle(x, y, w, h, fill = 'none', stroke = color, stroke_width = .3))
            
        width = (leftMargin - eps) // 2
        undoButton = (xmax, eps, width, eps)
        modeButton = (xmax, 3 * eps, width, eps)
        gridButton = (xmax, 5 * eps, width, eps)

        def edgesAppend(x1, y1, x2, y2):
            e = None
            if x1 < x2 or (x1 == x2 and y1 < y2):
                e = (x1, y1, x2, y2)            
            elif x1 != x2 or y1 != y2:
                e = (x2, y2, x1, y1)
            if e is not None:
                self.EDGES.append(e)
            return e

        def redraw(e, a):
            nonlocal lastChanges, mode, grid

            group.children.clear()

            if lastChanges:
                drawButton(undoButton, 'Undo')
            drawButton(modeButton, 'Mode', extText = 'Droite' if mode == LINE else 'Cercle')
            drawButton(gridButton, 'Grille', extText = 'Oui' if grid else 'Non')

            if grid:
                for i in range((ymax - eps - topMargin) // epsilon):
                    group.draw(draw.Line(0, topMargin + epsilon * i, xmax - eps, topMargin + epsilon * i, stroke_width = .1, stroke='black'))
                for j in range((xmax - eps) // epsilon + 1):
                    group.draw(draw.Line(epsilon * j, topMargin, epsilon * j, ymax - eps, stroke_width = .1, stroke='black'))

            for x1, y1, x2, y2 in e:
                line = draw.Line(x1, y1, x2, y2, stroke='green')
                group.draw(line)
            for x, y, r, _ in a:
                circle = draw.Ellipse(x, y, r, r, stroke='green', fill = 'none')
                group.draw(circle)
            for x, y in self.VERTICES:
                group.draw(draw.Circle(x, y, 4, stroke = 'green', fill='white' if y == ymax - eps else 'green'))                
                
        redraw(self.EDGES, self.ARCS)

        def drawCoords(x, y):
            group.draw(draw.Rectangle(xmax, 6.5 * eps, leftMargin, ymax, fill = 'white', stroke = 'none'))
            group.draw(draw.Text(f'{x:}', 12, xmax + eps, 7 * eps, stroke = 'black', stroke_width=.3, text_anchor='end'))
            group.draw(draw.Text(f'{y}', 12, xmax + eps, 7.5 * eps, stroke = 'black', stroke_width=.3, text_anchor='end'))

        widget = DrawingWidget(d)

        def forceGrid(u, coord):
            if coord == 0: # u = x
                return epsilon * round(u / epsilon) if grid else u
            else: # u = y
                return topMargin + epsilon * round((u - topMargin)/ epsilon) if grid else u


        @widget.mousedown
        def mousedown(widget, x, y, info):
            nonlocal a, b, last_edge, last_circle, lastChanges, mode, grid

            if x >= xmax:
                if x <= xmax + width:
                    if eps <= -y <= 2 * eps: # cancel last change
                        for change in lastChanges:
                            name = change[0]
                            data = change[1]
                            if name == 'verticesMinus':
                                del self.VERTICES[self.VERTICES.index(data)]
                            elif name == 'edgesPlus':
                                self.EDGES.append(data)
                            elif name == 'edgesMinus':
                                del self.EDGES[self.EDGES.index(data)]
                            elif name == 'arcsMinus':
                                del self.ARCS[self.ARCS.index(data)]
                            elif name == 'inArcMinus':
                                del self.ARCS[data][3][self.ARCS[data][3].index(change[2])]
                        lastChanges =[]
                    elif 3 * eps <= -y <= 4 * eps:
                        mode = 1 - mode
                    elif 5 * eps <= -y <= 6 * eps:
                        grid = not grid
            else:
                lastChanges =[]
                onpoint = False
                for u, v in self.VERTICES:
                    if (x - u) ** 2 + (y + v) ** 2 < 100:
                        newa, newb = u, v
                        onpoint = True
                        break
                if not onpoint:
                    if -y > ymax - eps - 7:
                        newa, newb = forceGrid(x, 0), ymax - eps
                    else:
                        d0 = 200
                        for i, (x1, y1, x2, y2) in enumerate(self.EDGES):
                            if (x - x1) * (x - x2) + (y + y1) * (y + y2) < 0:
                                u, v, w = y1 - y2, x2 - x1, x1 * y2 - x2 * y1
                                d = (u * x - v * y + w) ** 2 / (u ** 2 + v ** 2)
                                if d < d0:
                                    i0, x10, y10, x20, y20, u0, v0, w0, d0 = i, x1, y1, x2, y2, u, v, w, d
                        if d0 < 100:   
                            if abs(u0) < abs(v0):
                                newa, newb = x, round(- (u0 * x + w0) / v0)
                            else:
                                newa, newb = round((v0 * y - w0) / u0), -y
                            del self.EDGES[i0]
                            lastChanges.append(('edgesPlus', (x10, y10, x20, y20)))
                            e = edgesAppend(x10, y10, newa, newb)
                            if e is not None:
                                lastChanges.append(('edgesMinus', e))
                            e = edgesAppend(x20, y20, newa, newb) 
                            if e is not None:
                                lastChanges.append(('edgesMinus', e))
                        else:   
                            d0 = 100
                            for i, (xc, yc, r, l) in enumerate(self.ARCS):
                                d = abs(((xc - x) ** 2 + (yc + y) ** 2) ** .5 - r)
                                if d < d0:
                                    i0 , d0 = i, d
                            if d0 < 10:
                                xc, yc, r, _ = self.ARCS[i0] 
                                if abs(y + yc) > abs(x - xc):
                                    newa, newb = x, yc - round(((y + yc) / abs(y + yc)) * (r ** 2 - (x - xc) ** 2) ** .5)
                                else:
                                    newa, newb = xc +  round(((x - xc) / abs(x - xc)) * (r ** 2 - (y + yc) ** 2) ** .5), -y
                                self.ARCS[i0][3].append((newa, newb))
                                lastChanges.append(('inArcMinus', i0, (newa, newb)))
                            else:
                                newa, newb = forceGrid(x, 0), forceGrid(-y, 1)
                if last_edge:
                    e = edgesAppend(a, b, newa, newb)
                    if e is not None:
                        lastChanges.append(('edgesMinus', e))
                    last_edge = False
                if last_circle:
                    r = round(((x - a) ** 2 + (y + b) ** 2) ** .5)
                    if r > 5:
                        A = (x, -y, r, [(a, b)])
                        self.ARCS.append(A)
                        lastChanges.append(('arcsMinus', A))
                    last_circle = False
                else:
                    if not onpoint:
                        self.VERTICES.append((newa, newb))
                        lastChanges.append(('verticesMinus', (newa, newb)))
                a, b = newa, newb
                last_edge = False

            redraw(self.EDGES, self.ARCS)
            drawCoords(x, -y)
            widget.refresh()
            try:
                self.init()
            except:
                pass

        @widget.mousemove
        def mousemove(widget, x, y, info):
            nonlocal a, b, last_edge, last_circle
            if a is not None and (info['shiftKey'] or info['ctrlKey']):
                if mode == LINE:
                    last_edge = True
                    redraw(self.EDGES + [(a, b, x, -y)], self.ARCS)
                else:
                    last_circle = True
                    redraw(self.EDGES, self.ARCS + [(x, -y, ((x - a) ** 2 + (y + b) ** 2) ** .5,[])])
            drawCoords(x, -y)
            widget.refresh()

        return widget

    def makeTree(self):
    
        self.graph = defaultdict(list) # self.graph[i] = [(i1, j1), ..] où jk est une arête joignant i et ik
        for j in self.edges:
            (x1, y1, x2, y2), _ = self.edge[j]
            i1 = self.index_of_vertex[(x1, y1)]
            i2 = self.index_of_vertex[(x2, y2)]
            if i1 != i2: 
                self.graph[i1].append((i2, j))       
                self.graph[i2].append((i1, j))

        def index(i, i1): # indice dans self.graph[i] de (i1, j)
            k = 0
            while self.graph[i][k][0] != i1:
                k += 1
            return k

        self.parity = dict() # self.parity[i] = 0 s'il y a un nombre pair de boucles en i ou sinon 1
        for i in list(self.graph.keys()) + self.ground: 
            self.parity[i] = 0

        def updateParity(i, c):
            if c % 2:
                self.parity[i] = 1 - self.parity[i]

        # Remplacement des boucles par des arêtes fictives représentées par self.parity
        for i, (_, _, _, L) in enumerate(self.arc):
            if len(L) == 1:
                (x, y), _ = L[0]
                j = self.edge.index(((x, y, x, y),(i, 0, 0)))
                if j in self.edges:
                    i = self.index_of_vertex[(x, y)]
                    if i in self.parity.keys():
                       self.parity[i] = 1 - self.parity[i]
                    else:
                        self.parity[i] = 1

        def identify(i0, i):
            j0, j = self.parity[i0], self.parity[i]
            if j0 == 0:
                self.parity[i0] = j
            elif j == 1:
                self.parity[i0] = 0
            for i1, j in self.graph[i]:
                if i1 != i0:
                    self.graph[i0].append((i1, j))
                    k = self.graph[i1].index((i,j))
                    self.graph[i1][k] = (i0, j)
                else:
                    del self.graph[i0][self.graph[i0].index((i, j))]
                    updateParity(i0, 1)
            del self.graph[i]

        # identification des sommets du sol
        for i in self.ground:
            if i != self.root:
                identify(self.root, i)

        def collapse2Cycle():
            # suppression d'un cycle de longueur 2
            for i in list(self.graph.keys()):
                for i1, j1 in self.graph[i]:
                    for i2, j2 in self.graph[i1]:
                        if i2 == i and j1 != j2:
                            if i == self.root:
                                identify(i, i1)
                            else:
                                identify(i1, i)
                            return True
            return False

        def collapseCycle():                               
            # suppression d'un cycle s'il n'y a pas de cycle de longueur 2
            White, Grey, Black = 0, 1, 2
            color = {i : White for i in self.graph}
            father = {i : None for i in self.graph}
            i0 = -1
            def explore(i, iPred):
                nonlocal i0
                if color[i] == White:
                    color[i] = Grey
                    for i1, _ in self.graph[i]:
                        if i1 != iPred and i1 != i:
                            father[i1] = i
                            explore(i1, i)
                    color[i] =  Black
                elif color[i] == Grey:
                    i0 = i
                    raise Exception
            try:
                for i in self.graph:
                    explore(i, None)
            except:
                pass
            if i0 >= 0:
                cycle = [i0]
                i = father[i0]
                while i not in cycle:
                    cycle.append(i)
                    i = father[i]
                cycle = cycle[len(cycle) - list(reversed(cycle)).index(i) - 1:]
                if self.root in cycle:
                    i = cycle.index(self.root) + 1
                    cycle = cycle[i:] + cycle[:i]      
                for h in range(len(cycle) - 1):
                    identify(cycle[h + 1], cycle[h])
                return True
            else:
                return False
            
        # suppression des cycles
        hasCycle = True
        while hasCycle:
            while collapse2Cycle():
                pass
            hasCycle = collapseCycle()

        # self.graph et parity sont maintenant un arbre enraciné en self.root. Calcul d'une représentation adaptée de cet arbre
        self.tree = dict()
        def makeTree_(i, iPred):
            self.tree[i] = [(i1, j) for i1, j in self.graph[i] if i1 != iPred]
            for i1, _ in self.tree[i]:
                makeTree_(i1, i)
            j = self.parity[i]
            if j == 1:
                self.tree[i].append((-1, j))
        makeTree_(self.root, None)

    def g(self, i):
        if i == -1:
            return 1
        else:
            x = 0
            for i1, _ in self.tree[i]:
                x ^= self.g(i1)
            x += 1
            return x

    def playEdge(self, j):

        self.edges -= {j}

        def cc(i):

            mark = {k : False for k in self.vertices}
            def explore(k):
                if not mark[k]:
                    mark[k] = True
                    x, y = self.vertex[k]
                    for l in self.edges:
                        (x1, y1, x2, y2), _ = self.edge[l]
                        if (x1, y1) == (x, y):
                            explore(self.index_of_vertex[(x2, y2)])
                        elif (x2, y2) == (x, y):
                            explore(self.index_of_vertex[(x1, y1)])
            explore(i)
            return mark
        
        (x1, y1, x2, y2), _ = self.edge[j]
        if (x1, y1) != (x2, y2):
            i1 = self.index_of_vertex[(x1, y1)]
            i2 = self.index_of_vertex[(x2, y2)]
            mark1 = cc(i1)
            mark2 = cc(i2)
            for _ in range(2):
                i1, i2, mark1, mark2 = i2, i1, mark2, mark1
                s1 = {i for i in self.vertices if mark1[i]}
                s2 = {i for i in self.vertices if mark2[i]}
                #print(f'{s1=}\n{s2=}')
                if s1 & set(self.ground) and not (s2 & set(self.ground)):
                    self.vertices -= s2
                    for i in s2:
                        x, y = self.vertex[i]
                        for k in list(self.edges):
                            (x1, y1, x2, y2), _ = self.edge[k]
                            if (x1, y1) == (x, y) or (x2, y2) == (x, y):
                                self.edges -= {k}

        self.makeTree()
        self.G = self.g(self.root) - 1                      

    def play(self, hints = 2):
        """
        hints = 0 : aucune indication
                1 : calcul de G
                2 : calcul de G et mise en évidence des arêtes gagnantes si G != 0
        """
        
        self.init()

        maxX = 1
        for x, y in self.VERTICES:
            maxX = max(maxX, x)
        maxX += 50

        d = draw.Drawing(maxX, ymax)
        d.append(draw.Line(0, ymax - eps, maxX, ymax - eps, stroke_width = 1, stroke='green', stroke_dasharray='5,2'))
        Vertices_ = None
        Edges_ = None

        group = draw.Group()
        d.append(group)

        def play_(j):

            nonlocal Vertices_, Edges_
            
            Vertices_ = deepcopy(self.vertices)
            Edges_ = deepcopy(self.edges)
            self.playEdge(j)
            redraw()
            widget.refresh()

        def cancel():

            nonlocal Vertices_, Edges_
      
            if Vertices_ is not None:
                self.vertices = deepcopy(Vertices_)
                Vertices_ = None
                self.edges = deepcopy(Edges_)  
                self.makeTree()
                self.G = self.g(self.root) - 1
                redraw()
                widget.refresh()    


        def winningEdges():

            if self.G:
                Vertices = deepcopy(self.vertices)
                Edges = deepcopy(self.edges)
                J = []
                for j in Edges:
                    self.playEdge(j)
                    if self.G == 0:
                        J.append(j)
                    self.vertices = deepcopy(Vertices)
                    self.edges = deepcopy(Edges)  
                self.makeTree()
                self.G = self.g(self.root) - 1                  
                return J
            else:
                return []
            

        def redraw():

            J = winningEdges() if hints == 2 else []
            group.children.clear()
            for j in self.edges:
                color = 'red' if j in J else 'green'
                width = 2 if j in J else 1
                (x, y, x1, y1), u = self.edge[j]
                if u is None:
                    group.draw(draw.Line(x, y, x1, y1, stroke=color, stroke_width=width))
                else:
                    i, k, k1 = u
                    xc, yc, r, L = self.arc[i]
                    if len(L) == 1:
                        group.draw(draw.Circle(xc, yc, r, stroke = color, fill='none', stroke_width=width))
                    else:
                        _, t = L[k]
                        _, t1 = L[k1]
                        p = draw.Path(stroke=color, fill = 'none', stroke_width=width)
                        large_arc = t1 - t > math.pi if t1 > t else t - t1 < math.pi
                        group.draw(p.M(x, y).A(r, r, rot=0, large_arc=large_arc, sweep=1, ex=x1, ey=y1))
            for i in self.vertices:
                x, y = self.vertex[i]
                group.draw(draw.Circle(x, y, 4, stroke = 'green', fill='white' if y == ymax - eps else 'green'))
            if hints >= 1:
                for i in self.tree:
                    if i >= 0:
                        for i1, j in self.tree[i]:
                            (x, y, x1, y1), u = self.edge[j]
                            if i1 >= 0:
                                X = self.g(i1)
                                if X != 1:
                                    if u is None:
                                        U, V = (x+x1)//2, (y+y1)//2
                                    else:
                                        i, k, k1 = u
                                        xc, yc, r, L = self.arc[i]
                                        _, t = L[k]
                                        _, t1 = L[k1]
                                        if t1 < t: t1 += 2 * math.pi
                                        t0 = (t + t1) / 2       
                                        U, V = xc + r * math.cos(t0), yc + r * math.sin(t0)
                                    group.draw(draw.Text(f'{X}', 12, U, V))
                            else:
                                x, y =self.vertex[i] 
                                group.draw(draw.Circle(x, y, 4, fill='orange'))
                _, y = self.vertex[self.root]
                l = [self.vertex[i][0] for i in self.ground]
                x = sum(l) // len(l)
                group.draw(draw.Text(f'{self.G}', 14, x, y + 20, stroke = 'blue'))
                                 
        redraw()

        widget = DrawingWidget(d)

        @widget.mousedown
        def mousedown(widget, x, y, info):
            
            d0 = 200
            for j in self.edges:
                (x1, y1, x2, y2), none = self.edge[j] 
                if none is None and (x - x1) * (x - x2) + (y + y1) * (y + y2) < 0:
                    u, v, w = y1 - y2, x2 - x1, x1 * y2 - x2 * y1
                    d = (u * x - v * y + w) ** 2 / (u ** 2 + v ** 2)
                    if d < d0:
                        j0, d0 = j, d
            if d0 < 100: 
                play_(j0)
            else:
                d0 = 100
                for j in self.edges:
                    (x1, y1, x2, y2), u = self.edge[j] 
                    if u is not None:
                        i, _, _ = u
                        xc, yc, r, _ = self.arc[i]
                        d = abs(((xc - x) ** 2 + (yc + y) ** 2) ** .5 - r)
                        if d < d0:
                            i0, d0 = i, d
                if d0 < 10:
                    xc, yc, r, L = self.arc[i0]
                    if len(L) == 1:
                        (x0, y0), _ = L[0]
                        play_(self.edge.index(((x0, y0, x0, y0),(i0, 0, 0))))
                    else:
                        d = ((xc - x) ** 2 + (yc + y) ** 2) ** .5  + x - xc
                        t = 2 * math.atan((-y - yc) / d) if d else math.pi
                        k0 = len(L) - 1
                        for k in range(len(L) - 1):
                            _, t1 = L[k]
                            _, t2 = L[k+1]
                            if t1 <= t < t2:
                                k0 = k
                                break
                        (x1, y1), _ = L[k0]
                        k1 = (k0 + 1) % len(L)
                        (x2, y2), _ = L[k1] 
                        play_(self.edge.index(((x1, y1, x2, y2),(i0, k0, k1))))
                else:
                    cancel()
                
        return widget


<IPython.core.display.Javascript object>

### Jouer avec/contre l'ordinateur

Pour définir une figure :  
$H\texttt{= HACKENBUSH(}\!\!$ *name* $\!\!\texttt{)}$, où *name* est une figure enregistrée sur disque dans $\texttt{exemplesHackenbush/}\!
\!$ *name* $\!\!\texttt{.json}$.

Pour jouer :  
$H\texttt{.play(}\!\!$ *hints* $\texttt{= 2)}$

*hints* $=0$ : aucune aide n'est apportée,  
*hints* $=1$ : les tensions sont indiquées, ainsi que la valeur de la figure,  
*hints* $=2$ : idem *hints* $=1$ et, de plus, si la valeur est non nulle, les arêtes correspondant aux mouvements gagnants sont indiquées en rouge.


Cliquer sur l'arête à supprimer.

Cliquer à un endroit sans arête pour annuler le dernier coup.

In [10]:
HACKENBUSH('girlWithUmbrella').play(0)

DrawingWidget()

In [3]:
HACKENBUSH('girlWithDog').play(1)

DrawingWidget()

In [4]:
HACKENBUSH('girls').play()

DrawingWidget()

In [9]:
HACKENBUSH('TheHackenbushEstate').play()


DrawingWidget()

### Construction d'une figure

ou ajout de nouveaux éléments à une figure existante

Pour tracer  
- un sommet : cliquer  
- un segment (arête rectiligne) : en mode "Droite", 
  1. garder la touche SHIFT (ou CTRL) appuyée,
  2. cliquer pour définir une extrémité,
  3. bouger la souris,
  4. cliquer pour définir l'autre extrémité,
  5. relacher la touche SHIFT (ou CTRL) (ou, pour tracer un nouveau segment issu de la dernière extrémité, aller en 3.)
- un cercle (arête boucle) : en mode "Cercle",
  1. garder la touche SHIFT (ou CTRL) appuyée,
  2. cliquer pour définir le sommet situé sur le cercle (l'extrémité de la boucle),
  3. bouger la souris,
  4. cliquer pour définir le centre du cercle,
  5. relacher la touche SHIFT (ou CTRL)


Attention : si on fait passer un segment $AB$ ou un cercle d'extrémité $A$ par un point $C$ déja tracé, ce point $C$ ne sera pas considéré comme un sommet partageant  en deux arêtes le segment ou le cercle. Pour cela, il faut tracer le segment $AB$ ou le cercle d'extrémité $A$ puis tracer $C$ sur le segment ou le cercle; ou alors, dans le cas d'un segment, tracer le segment $AC$ puis le segment $CB$.

Si l'on trace un cercle et son extrémité, et si, ensuite on trace $n$ sommets sur ce cercle, on obtient un cycle formé de $n+1$ arcs du cercle. On ne peut pas ne conserver que certains de ces arcs ...

In [6]:
H = HACKENBUSH()
H.modify()

DrawingWidget()

et sauvegarde de la figure construite

In [7]:
H.save('essai')

In [ ]:
HACKENBUSH('essai').play()

### Préliminaires à la démonstration du théorème de Hackenbush

#### Représentation binaire d'un entier négatif

La représentation binaire d'un entier positif ou nul $x=\Sigma_0^\infty x_i2^i$ ($x_i\in\{0,1\}$ nuls APCR) est la suite $(x_i)_{i\geqslant0}$ et on note $x=\overline{\ldots x_1x_0}$.   
Pour $a\in\{0,1\}$, soit $\widetilde a:=1-a$ et définissons la représentation binaire de $-x-1<0$ par
la suite  $(\widetilde{x_i})_{i\geqslant0}$.  
On écrit donc $\forall x\geqslant0,\,-x-1=\overline{\ldots\widetilde{x_1}\widetilde{x_0}}$ ; on pose $\widetilde x:=-x-1=\overline{\ldots\widetilde{x_1}\widetilde{x_0}}$ ($\widetilde{x_i}=1$ APCR) et $\widetilde{\widetilde x}:=x$.  
Ainsi $\forall x\in\mathbf Z,\,-x=\widetilde{x-1}$. Par exemple, $-1=\widetilde 0=\widetilde{\overline{\ldots 00}}=\overline{\ldots 11}$.

Cette définition n'aurait pas d'intérêt si on n'avait pas la

**Prop. (opérations sur les entiers)**

La représentation binaire de la somme ou de la différence de deux entiers relatifs s'obtient par l'algorithme usuel (chiffre par chiffre, retenue éventuelle, etc.) appliqué aux représentations binaires des deux entiers.

**preuve**

Remarque préliminaire : si $0\leqslant x=\overline{x_{k-1}\ldots x_0}<2^k$, alors
$2^k-x-1=(2^k-1)-x=\overline{1\ldots 1}-\overline{x_{k-1}\ldots x_0}=\overline{\widetilde{x_{k-1}}\ldots\widetilde{x_0}}$.

Premier cas : somme $y+(-x)=y-x$ d'un entier $y\geqslant0$ et d'un entier $-x<0$.

Soit $k$ tq $x$ et $y$ sont $<2^k$, alors $x=\overline{x_{k-1}\ldots x_0}$ et $y=\overline{y_{k-1}\ldots y_0}$

D'après la remarque, $2^k-x=\overline{\widetilde{z_{k-1}}\ldots\widetilde{z_0}}$ où $z:=x-1=\overline{z_{k-1}\ldots z_0}$, donc   
$2^k+y-x=y+(2^k-x)=\overline{y_{k-1}\ldots y_0}+\overline{\widetilde{z_{k-1}}\ldots\widetilde{z_0}}=\overline{t_kt_{k-1}\ldots t_0}$
(algorithme usuel).

- si $t_k=1$, $y-x=\overline{t_{k-1}\ldots t_0}=\overline{\ldots00t_{k-1}\ldots t_0}$,
- si $t_k=0$, $2^k+y-x=\overline{t_{k-1}\ldots t_0}$ et, en appliquant de nouveau la remarque préliminaire,   
$y-x=-(2^k-t)=\widetilde{2^k-t-1}=\widetilde{\overline{\widetilde{t_{k-1}}\ldots\widetilde{t_0}}}=\overline{\ldots11t_{k-1}\ldots t_0}$.

Dans les deux cas, on voit que $y-x$ admet bien l'écriture binaire obtenue par l'algorithme usuel appliqué à l'addition
de $y=\overline{\ldots00y_{k-1}\ldots y_0}$ et de $-x=\widetilde z=\widetilde{\overline{\ldots00z_{k-1}\ldots z_0}}=\overline{\ldots11\widetilde{z_{k-1}}\ldots\widetilde{z_0}}$

Autres cas : même méthode  $\qquad \blacksquare$

En particulier $2x=x+x=\overline{\ldots x_1x_00}$ donc $2^2x=\overline{\ldots x_1x_000}$, etc.  
$2^k$ divise $x$ ssi $\forall i<k,\,x_i=0$   
et aussi, $\forall x\in\mathbf Z,k\in\mathbf N, \,x= 2^k\times\overline{\ldots x_{k+1}x_k}+\overline{x_{k-1}\ldots x_0}$.

#### Prolongement à $\mathbf Z$ de la nim-addition

Pour $x=\overline{\ldots a_1a_0}\in \mathbf Z$ et $y=\overline{\ldots b_1b_0}\in \mathbf Z$, on pose $x\oplus y:=\overline{\ldots (a_1\oplus b_1)(a_0\oplus b_0)}$
ce qui fait de $(\mathbf Z,\oplus)$ un groupe commutatif de neutre $0$ et $\ominus x=x$.  
Ainsi $-1\oplus x=\overline{\ldots11}\oplus x=\widetilde x=-x-1$ et par exemple $-3\oplus7=\widetilde2\oplus7=\widetilde5=-6$.

#### La fonction amitié

On définit l'*amitié* de deux entiers $x,y\in\mathbf Z$ par 
$$(x|y):=\begin{cases}
2^{n+1}-1 \text{ où } n=\max\{k\geqslant0\,,\,x\equiv y \text{ mod } 2^k\} &\text{si } x\neq y\\
-1 &\text{si } x=y
\end{cases}
$$

**Prop. (invariance)**

$\forall x,y,z\in \mathbf Z,\; (x|y)=(x+z|y+z)=(x\oplus z|y\oplus z)\qquad\blacksquare$

**preuve**  
La première égalité est évidente.   
Pour la deuxième, il faut remarquer que  $x\equiv y \text{ mod } 2^k$ signifie que les $k$ derniers chiffres binaires de $x$ sont les mêmes que ceux de $y$  $\qquad\blacksquare$ 

**Prop.**

$x\oplus(x|0)=x-1$, $\qquad y\oplus(y|-1)=y+1\quad$ et $\quad x\oplus y\oplus(x|y)=x\oplus y -1$.

**preuve**

1. Si $x=0$, $0\oplus(0|0)=0\oplus(-1)=-1=0-1$.  
Si $x\neq0$ $(x|0)=2^{n+1}-1=\overline{\ldots0011\ldots1}$ ($n+1$ occurences de $1$) et, par définition de $n$, $x=\overline{\ldots x_{n+1}10\ldots0}$
donc $x\oplus(x|0)=\overline{\ldots x_{n+1}01\ldots1}=x-1$.
2. s'obtient à partir de 1. où $x=y+1$ : $(x|0)=(y|-1)$ par invariance.
3. Par invariance et d'après 1., $x\oplus y\oplus(x|y)=x\oplus y\oplus(x\oplus y|0)=x\oplus y -1$ $\qquad\blacksquare$

**Prop. (des trois amis)**

Si $x, y,z$ sont trois entiers distincts, deux des trois nombres $(x|y), (y|z), (z|x)$ sont égaux et strictement plus petits que le troisième. 

**preuve**

Supposons $(x|y)\geqslant (y|z)\geqslant (z|x)$ de sorte que $x-y=2^nu$, $z-y=2^pv$ et $x-z=2^qw$ avec $n\geqslant p\geqslant q$ et $u,v,w$ impairs.  
$2^nu=2^pv+2^qw$ donne $2^{n-q}u=2^{p-q}v+w$ donc, par l'absurde, $n >q$ puis $p=q$ $\qquad\blacksquare$

### Démonstration du théorème de Hackenbush

Cette démonstration ne fait que reprendre celle de Conway mais en précisant certains points qui sont évidents pour Conway mais qui ne l'étaient pas pour moi $\ldots$

On fixe une position du jeu $G$ et son arbre enraciné associé $T$.

La *charge* d'une arête $a$ de $G$ ou de $T$ est l'ensemble ($a$ non compris) des arêtes qui doivent être enlevées si on supprime l'arête $a$.

A toute arête $a$ de $T$ on associe son *poids*
$\omega(a)=(\tau(a)|0)$.

**Théorème des poids**

Dans l'arbre $T$, la tension d'une arête $a$ est la nim-somme des poids de cette arête et de toutes les arêtes de sa charge.

**preuve**
Toute arête de la charge de $a$ est soit une arête fille de $a$, soit une arête de la charge de l'une de ces arêtes $b$. Donc, par induction

$\displaystyle
\bigoplus_{c \,\in\text{ charge}(a)}\omega(c)=
\bigoplus_{b\text{ fille de }a}\tau(b)=\tau(a)-1
$ ;
donc
$\displaystyle
\omega(a)\oplus\bigoplus_{c \,\in\text{ charge}(a)}\omega(c)=
(\tau(a)|0)\oplus(\tau(a)-1)=\tau(a)
$ 
$\qquad\blacksquare$

**corollaire**

Le poids de $G$ (ou de $T$) est la nim-somme des poids des arêtes de $T$ $\qquad\blacksquare$

**Théorème du changement de prise**

Soient $G$ et  $G'$ deux figures et $a$ et $a'$ des arêtes de $T$ et $T'$ respectivement.  
On suppose que la charge de $a$ dans $G$ et celle de $a'$ dans $G'$ sont isomorphes.  
Alors $\omega_G(a)=\omega_{G'}(a')$.

Par exemple, dans la figure $\texttt{girls}$, on voit que le bras de la fille de gauche et le pied de la fille de droite qui tiennent leur parapluie ont pour tension $6$ et $2$ respectivement. On a bien $(6|0)=3=(2|0)$.

**preuve**

Comme la charge $C$ de $a$ dans $G$ ou de $a'$ dans $G'$ est connexe, on peut supposer que les extrémités $s$ et $s'$ de $a$ et $a'$ qui sont dans $C$ sont reliées par une arête $b$ de $C$.

Si $b$ participe à un cycle de $C$, les sommets $s$ et $s'$ sont identifiés dans $T$ ou $T'$, $\tau_G(a)=\tau_{G'}(a')$ donc $\omega_G(a)=\omega_{G'}(a')$. 

Sinon la suppression de $b$ partage $C$ en deux composantes $C_s$ et $C_{s'}$. La nim-somme $x$ (resp. $x'$) des tensions des arêtes de $C_s$ (resp. $C_{s'}$) incidentes à $s$ (resp. $s'$) est la même dans $G$ et dans $G'$.   
On notera que $x$ est le poids de $s$ ou de $C_s$ (à ne pas confondre avec la notion de poids d'une arête).

![chgtPrise](images/chgtPrise.png)

On a $\tau_G(a)=((x'+1)\oplus x)+1$ et, par invariance de la fonction amitié,   
$\omega_G(a)=(((x'+1)\oplus x)+1|0)=((x'+1)\oplus x|-1)=(x'+1|-1\ominus x)=(x'+1|- x-1)=(x+1|- x'-1)=\omega_{G'}(a')$ $\qquad\blacksquare$

**Lemme**

Si $a$ est une arête de $T$ et $b$ une arête qui *supporte* $a$ (i.e. $b$ se trouve sur le chemin de la racine à $a$), si on note
$\tau_a(b)$ la tension de $b$ dans l'arbre $T_a$ obtenu en supprimant l'arête $a$ (et sa charge), alors
$
(\tau(b)|\tau_a(b))=\omega(a)
$.

**preuve**

Comme $T$ s'obtient à partir de $T_a$ en lui rajoutant l'arête $a$ et sa charge, on voit de proche en proche que $\tau(b)=f(\tau(a))$ et $\tau_a(b)=f(0)$
où $f$ est de la forme   
$f(x)=((\ldots((((x \oplus y)+1)\oplus z)+1)\ldots )\oplus t) +1$.  
Alors, par invariance de la fonction amitié,  
$(\tau(b)|\tau_a(b))= (f(\tau(a))|f(0))=(\tau(a)|0)=\omega(a)\qquad\blacksquare$

Un  *cycle* est un ensemble d'arêtes formant un circuit ou joignant un élément du sol à un autre.  
On dit que deux arêtes sont *concycliques* si l'ensemble non vide des cycles contenant la première arête coïncide avec  l'ensemble des cycles contenant la deuxième arête.  
Il s'agit d'une relation d'équivalence. Dans la figure suivante, chaque classe est indiquée par une couleur différente (mais les classes réduites à un singleton, ainsi que l'arête verticale qui n'appartient à aucun cycle sont indiquées en noir).

![hFig0](images/hFig0.png)

Sur l'ensemble des arêtes d'une classe de concyclicité donnée, on définit $(a|b)=(\tau_a(b)|0)$ le poids de $b$ dans la figure $G_a$ obtenue à partir de $G$ en
supprimant l'arête $a$ ; cela a bien un sens car, dans $G_a$, il n'y a pas de cycle contenant $b$ (un tel cycle devrait contenir $a$).

D'après le théorème du changement de prise, on a $(a|b)=(b|a)$.

**Théorème des trois arêtes concycliques**

Si $a,b,c$ sont trois arêtes concycliques distinctes, deux des trois nombres $(a|b), (b|c), (c|a)$ sont égaux et strictement plus petits que le troisième.

**preuve**

Soit un cycle $C$ contenant $a$, $b$ et $c$. On suppose que, dans $C$, $b$ est situé entre $a$ et $c$. En appliquant le lemme à la figure $G_c$ on obtient  
$(\tau_c(a)|\tau_{cb}(a))=\omega_c(b)=(b|c)$. Or les arêtes supportées par $a$ dans $G_b$ et dans $G_{cb}$ sont les mêmes donc $\tau_{cb}(a)=\tau_{b}(a)$ et
$(\tau_c(a)|\tau_b(a))=(b|c)$.  
Or $(\tau_b(a)|0)=(a|b)$ et $(\tau_c(a)|0)=(c|a)$, donc le résultat se déduit de la propriété des trois amis 
($\tau_b(a)\neq\tau_c(a)$ car, dans $G_c$ le poids de la figure formée par $b$ et sa charge est $>0$ : voir la démonstration du lemme) $\qquad\blacksquare$

**preuve du théorème de Hackenbush**

On montre par induction, ou par récurrence sur le nombre d'arêtes de $G$, que $g(G)=\omega(G)$.

1. Cas où $G$ n'a pas d'arête, $g(G)=0=\omega(G)$.

2. Cas où une unique arête $a$ de $G$ a une extrémité $s$ au sol et où l'autre extrémité $s'$
   est différente de $s$ ($a$ n'est pas une boucle).    
   ![hFig3](images/hFig3.png)   
   Si $G$ est réduit à l'arête $a$, $g(G)=1=\omega(G)$.   
   Sinon on note $\overline G$ le graphe obtenu à partir de $G$ en supprimant l'arête $a$. On fait de 
   $\overline G$ une figure en l'attachant au sol par $s'$.    
   On applique l'HR (hypothèse de récurrence) à $\overline{G}$, $G_b$ et $\overline{G_b}$ pour $b$ arête de $\overline{G}$ :
   $$
    \begin{align*}
    g(G)&=\text{mex}\{g(G_b)\,|\,b \text{ arête de } G\}\\
        &=\text{mex}(\{0\}\cup\{g(G_b)\,|\,b \text{ arête de } G, b\neq a\}) \quad G_a \text{n'a pas d'arête}\\
        &=\text{mex}(\{0\}\cup\{\omega(G_b)\,|\,b \text{ arête de } \overline G\})\\
        &=\text{mex}(\{0\}\cup\{1+\omega(\overline{G_b})\,|\,b \text{ arête de } \overline G\})\\
        &=\text{mex}(\{0\}\cup\{1+g(\overline{G_b})\,|\,b \text{ arête de } \overline G\})\\
        &=1+\text{mex}\{g(\overline{G_b})\,|\,b \text{ arête de } \overline G\}\\
        &=1+g(\overline G)\\
        &=1+\omega(\overline G)\\
        &=\omega(G)
    \end{align*}
   $$
3. Cas où chaque arête de $G$ est élément d'un cycle. Dans ce cas l'arbre $T$ est réduit à $n$ brins issus de la racine, un brin par arête de $G$.  
Si $a$ est une arête de $G$, soit $a_0=a,a_1,\ldots,a_k$ la classe $C$ des arêtes concycliques à $a$. D'après le corollaire du théorème des poids, le poids de $G_a$ est la nim-somme des poids des arêtes de $G_a$, ce qui donne   
$\displaystyle
\omega(G_a)=(a|a_1)\oplus\ldots\oplus(a|a_k)\oplus\bigoplus_{b \text{ arête de } G\setminus C }1
$  
(en effet, une arête $b$ qui n'est pas concyclique à $a$ est membre d'un cycle de $G_a$).   
Cette nim-somme contient $n-1$ termes impairs et a donc une parité différente de $\displaystyle\omega(G)=\bigoplus_{b \text{ arête de } G }1
=0$ ou $1$ suivant que $n$ est pair ou impair.  
En particuliers, $\omega(G)\neq\omega(G_a)=g(G_a)$ (HR).  
  - Si $\omega(G)=0$, les $\omega(G_a)$ sont $>0$ et leur mex  $g(G)$ est $0=\omega(G)$.
  - Si $\omega(G)=1$, les $\omega(G_a)$ sont $\neq1$ et leur mex  $g(G)$ sera égal à $1=\omega(G)$ si on montre qu'il existe un $a$ tq 
$\omega(G_a)=0$ :   
Comme $n$ est impair il existe une classe concyclique $C$ comportant un nombre impair d'arêtes.   
Pour ce $C$, la nim-somme $\displaystyle\bigoplus_{b \text{ arête de } G\setminus C }1$ est nulle.   
Soient $a_1,a_2\in C$ tq $(a_1|a_2)$ soit maximum,   
soient $a_3,a_4\in C\setminus\{a_1,a_2\}$ tq $(a_3|a_4)$ soit maximum,  
et ainsi de suite.  
A la fin du procédé, $C\setminus\{a_1,a_2\,\ldots,a_{2h}\}$ est réduit à une arête $a$.  
Pour ce $a$, le théorème des trois arêtes concycliques montre que $(a|a_{2i-1})=(a|a_{2i})$ pour tout $i$ de
sorte que la nim-somme $(a|a_1)\oplus\ldots\oplus(a|a_{2h})$ est nulle et que $\omega(G_a)$ est bien nul.
4. Cas général.  
Considérons les $k$ arêtes de $T$ issues de la racine et qui ne sont pas des brins correspondant à des cycles.  
Elles correspondent à $k$ arêtes $a_1,\ldots,a_k$ de $G$.   
Pour $i=1,\ldots,k$, notons $G_i$ la figure constituée de $a_i$ et de sa charge attachée au sol par l'extrémité de $a_i$ qui n'est pas dans la charge.   
Dans la figure suivante, $a_1$ est le tronc du pommier et $G_1$ le pommier lui-même.  
![hFig1](images/hFig1.png)  
On note $G_0$ la figure constituée des arêtes de $G$ qui ne sont pas des arêtes d'un $G_i$. Ici :  
![hFig2](images/hFig2.png)  
Si l'on supprime une arête $a$ de $G_i$, $i=0,\ldots,k$ les arêtes des $G_j$, $j\neq i$, sont toutes conservées donc les
$G_j$ (cas 2. et 3.) sont inchangés alors que $G_i$ est remplacé par $G_{i,a}$.  
Donc, pour ce mouvement, le jeu $G$ se comporte comme la somme des jeux $G_i$, $i=0,\ldots,k$ et, en appliquant les cas 2. et 3.,   
$g(G)=\oplus_i g(G_i)=\oplus_i\omega(G_i)=\omega(G)\qquad\blacksquare$